In [ ]:
import os
import math
import datetime
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
from rasterio.env import Env

# ------------------------------- CONFIGURATION ------------------------------- #

INPUT_PARENT_FOLDER = "upload/TrainingData"
EARTH_SUN_EXCEL_FILE = "upload/TOA_help/Earth_Sun_distance.xlsx"
OUTPUT_PARENT_FOLDER = "Output"
APPLY_DOS = False

# ------------------------- HELPER FUNCTIONS ------------------------- #

# Function to read the metadata 
def read_metadata(input_folder):
    metadata_path = os.path.join(input_folder, 'BAND_META.txt')
    if not os.path.exists(metadata_path):
        raise FileNotFoundError(f"Metadata file not found at: {metadata_path}")
    else: 
        print(f"Reading metadata file from : {metadata_path}")

    metadata = {}
    with open(metadata_path) as f:
        for line in f:
            key, value = line.split('=')
            metadata[key.strip()] = value.strip()

    return metadata

# Function to get the distances between earth and sun
def get_earth_sun_distance(date, excel_path):
    df = pd.read_excel(excel_path, sheet_name="Earth_sun_distance", header=1)
    match = df.where(df == date).stack()
    if match.empty:
        raise ValueError(f"Date {date.strftime('%d-%b-%Y')} not found in Excel file.")

    row_idx, col_label = match.index[0]
    col_idx = df.columns.get_loc(col_label)
    return df.iloc[row_idx, col_idx + 2]

# Function to get esun_values
def get_esun_values(sat_id):
    esun = {
        'RS2': {'B2': 185.33, 'B3': 157.766, 'B4': 111.359},
        'RS2A': {'B2': 185.347, 'B3': 158.262, 'B4': 110.81}
    }
    key = 'RS2' if sat_id == 'IRS-R2' else 'RS2A'
    return esun[key]

# Function to read bands
def read_bands(input_folder):
    with Env(GTIFF_SRS_SOURCE='GEOKEYS'):
        return [(rxr.open_rasterio(os.path.join(input_folder, f"BAND{i}.tif")).where(lambda x: x != 0, np.nan)) for i in [2, 3, 4]]

# Function to convert DN values to radiance
def dn_to_radiance(dn, lmin, lmax):
    return dn * (lmax - lmin) / 1024

# Function to convert radiance to reflectance
def radiance_to_reflectance(radiance, esun, d_es, sun_elevation):
    return (math.pi * radiance * (d_es ** 2)) / (esun * math.sin(sun_elevation))

# Function for DOS
def apply_dark_object_subtraction(ref_bands, wavelengths):
    red = ref_bands[1].values
    dark = next((px for px in np.unique(red[~np.isnan(red)])[:100] if px > 0), None)
    if dark is None:
        raise ValueError("No valid dark object found.")

    green_median = np.nanmedian(ref_bands[0].values)
    nir_median = np.nanmedian(ref_bands[2].values)

    model = -4 if nir_median < 0.02 else (-1 if green_median > 0.15 else -2)
    multiplier = 0.8 if dark < 0.01 else (1.2 if dark > 0.05 else 1.0)

    haze = [(dark * ((wavelengths[i] / wavelengths['BAND3']) **model) * multiplier) for i in ['BAND2', 'BAND3', 'BAND4']]
    return [(ref - h).clip(0, 1.0) for ref, h in zip(ref_bands, haze)]

# ------------------------- MAIN FUNCTION ------------------------- #

# Function to convert DN to TOA reflectance
def convert_DN_to_reflectance(input_folder, excel_file, output_folder, apply_dos):
    metadata = read_metadata(input_folder)
    scene_id = metadata['OTSProductID']
    date = datetime.datetime.strptime(metadata['DateOfPass'], '%d-%b-%Y').replace(year=2020)
    d_es = get_earth_sun_distance(date, excel_file)
    esun = get_esun_values(metadata['SatID'])
    sun_elev = math.radians(float(metadata['SunElevationAtCenter']))

    # Read DN values and convert to radiance
    b2_dn, b3_dn, b4_dn = read_bands(input_folder)
    b2_rad = dn_to_radiance(b2_dn, float(metadata['B2_Lmin']), float(metadata['B2_Lmax']))
    b3_rad = dn_to_radiance(b3_dn, float(metadata['B3_Lmin']), float(metadata['B3_Lmax']))
    b4_rad = dn_to_radiance(b4_dn, float(metadata['B4_Lmin']), float(metadata['B4_Lmax']))

    del b2_dn, b3_dn, b4_dn

    # Convert radiance to reflectance
    b2_ref = radiance_to_reflectance(b2_rad, esun['B2'], d_es, sun_elev)
    b3_ref = radiance_to_reflectance(b3_rad, esun['B3'], d_es, sun_elev)
    b4_ref = radiance_to_reflectance(b4_rad, esun['B4'], d_es, sun_elev)

    del b2_rad, b3_rad, b4_rad

    reflectance_bands = [b2_ref, b3_ref, b4_ref]

    del b2_ref, b3_ref, b4_ref

    if apply_dos:
        wavelengths = {'BAND2': 0.555, 'BAND3': 0.650, 'BAND4': 0.815}
        reflectance_bands = apply_dark_object_subtraction(reflectance_bands, wavelengths)

    # Stack and export
    scene = xr.concat(reflectance_bands, dim='band').assign_coords(band=['BAND2', 'BAND3', 'BAND4'])
    scene.name = scene_id

    del reflectance_bands

    os.makedirs(output_folder, exist_ok=True)
    output_file = os.path.join(output_folder, f"{scene_id}.tif")

    print(f"Saving output to: {output_folder}")
    scene.to_dataset('band').rio.to_raster(
        output_file, driver='COG', dtype='float32', nodata=0,
        blockxsize=256, blockysize=256, tiled=True,
        compress='deflate', quality=1, predictor=3,
        BIGTIFF='IF_NEEDED', overviews='auto', windowed=True,
        num_threads='all_cpus')

# ------------------------- EXECUTION ------------------------- #

input_dirs = [os.path.join(dp, d) for dp, dns, _ in os.walk(INPUT_PARENT_FOLDER) for d in dns]

for idx, input_folder in enumerate(input_dirs, 1):
    output_folder = os.path.join(OUTPUT_PARENT_FOLDER, os.path.basename(os.path.dirname(input_folder)))
    print(f"Processing folder {idx}/{len(input_dirs)}: {input_folder}")
    convert_DN_to_reflectance(input_folder, EARTH_SUN_EXCEL_FILE, output_folder, APPLY_DOS)
    print(f"Folder {idx} processed successfully. Output saved to: {output_folder}")

print("TOA reflectance conversion successful for all the scenes.")